# Sketch-to-Story Colab Runner

이 노트북은 GitHub `Colab` 브랜치 기준 실행본입니다. `inputs/` 아래 이야기 폴더 하나를 선택한 뒤 A, B, C, D, E, F를 각각 독립 셀로 실행합니다. 피드백은 재생성 없이 평가/기록용으로 저장합니다.


## 1. 환경 확인

Colab 메뉴에서 `런타임 > 런타임 유형 변경 > GPU`를 선택한 뒤 실행하세요. C/D/E/F는 Qwen2.5-VL-3B를 사용하므로 CPU 실행은 권장하지 않습니다.


In [ ]:
import os
import sys
import shutil
from pathlib import Path

try:
    import torch
    print('Python:', sys.version.split()[0])
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
except Exception as exc:
    print('torch check skipped:', exc)

print('Working directory:', Path.cwd())
print('Disk usage:', shutil.disk_usage('/content') if Path('/content').exists() else shutil.disk_usage('.'))


## 2. 의존성 설치

Colab 기본 torch는 유지하고 나머지 패키지만 설치합니다. 설치 후 런타임 재시작 안내가 나오면 재시작한 뒤 다음 셀부터 다시 실행하세요.


In [ ]:
%pip install -q -r requirements-colab.txt


## 3. 프로젝트 경로와 이야기 폴더 선택

GitHub에서 clone한 저장소 루트에서 이 노트북을 실행한다고 가정합니다. 드롭다운에서 실행할 이야기 폴더를 선택한 뒤 다음 셀을 실행하세요.


In [ ]:
from pathlib import Path
import sys
import re
import json
from IPython.display import display
import ipywidgets as widgets

PROJECT_DIR = Path.cwd().resolve()
INPUTS_DIR = PROJECT_DIR / 'inputs'
OUTPUT_ROOT = PROJECT_DIR / 'outputs'
sys.path.insert(0, str(PROJECT_DIR))

def natural_key(path: Path):
    return [int(part) if part.isdigit() else part for part in re.split(r'(\d+)', path.name)]

STORY_DIRS = sorted([p for p in INPUTS_DIR.iterdir() if p.is_dir()], key=natural_key)
if not STORY_DIRS:
    raise FileNotFoundError(f'No story folders found in {INPUTS_DIR}')

story_dropdown = widgets.Dropdown(
    options=[(p.name, p.name) for p in STORY_DIRS],
    value=STORY_DIRS[0].name,
    description='이야기:',
    layout=widgets.Layout(width='520px'),
)
display(story_dropdown)


In [ ]:
SELECTED_STORY = story_dropdown.value
STORY_INPUT_DIR = INPUTS_DIR / SELECTED_STORY
STORY_OUTPUT_ROOT = OUTPUT_ROOT / SELECTED_STORY
STORY_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

IMAGE_EXTENSIONS = {'.png', '.jpg', '.jpeg'}
IMAGE_FILES = sorted([p for p in STORY_INPUT_DIR.iterdir() if p.suffix.lower() in IMAGE_EXTENSIONS], key=natural_key)
if not IMAGE_FILES:
    raise FileNotFoundError(f'No images found in {STORY_INPUT_DIR}')

print('Selected story:', SELECTED_STORY)
print('Input directory:', STORY_INPUT_DIR)
print('Output root:', STORY_OUTPUT_ROOT)
print('Images:', [p.name for p in IMAGE_FILES])


## 4. 이미지 미리보기

선택한 이야기 폴더의 그림 순서를 확인합니다.


In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

cols = 5
rows = (len(IMAGE_FILES) + cols - 1) // cols
plt.figure(figsize=(cols * 2.4, rows * 2.8))
for idx, image_path in enumerate(IMAGE_FILES, start=1):
    image = Image.open(image_path).convert('RGB')
    ax = plt.subplot(rows, cols, idx)
    ax.imshow(image)
    ax.set_title(image_path.name)
    ax.axis('off')
plt.tight_layout()
plt.show()


## 5. 공통 import와 메모리 정리 함수


In [ ]:
import gc

from pipeline_a import run_experiment_a, run_sequence_story
from run_experiments_cd_qwen3b import run_selected_experiments

def clear_runtime_memory():
    gc.collect()
    try:
        import torch
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            print('CUDA cache cleared')
    except Exception as exc:
        print('CUDA cleanup skipped:', exc)

def experiment_dir(name: str) -> Path:
    out = STORY_OUTPUT_ROOT / name.upper()
    out.mkdir(parents=True, exist_ok=True)
    return out


## 6. Run A

A는 선택한 이야기 폴더의 모든 이미지를 각각 처리합니다. 빠른 Colab 실행을 위해 기본 backend는 `structured_template`입니다.


In [ ]:
A_OUTPUT_DIR = experiment_dir('A')
A_RESULTS = []
for image_path in IMAGE_FILES:
    print(f'[A] {image_path.name}')
    result = run_experiment_a(
        str(image_path),
        output_dir=str(A_OUTPUT_DIR),
        clip_threshold=0.22,
        story_backend='structured_template',
    )
    A_RESULTS.append(result)
print(f'A complete: {A_OUTPUT_DIR}')


## 7. Run B

B는 선택한 이야기 폴더의 1~10번 이미지를 하나의 순서형 이야기로 묶습니다.


In [ ]:
B_OUTPUT_DIR = experiment_dir('B')
B_RESULT = run_sequence_story(
    image_dir=str(STORY_INPUT_DIR),
    output_dir=str(B_OUTPUT_DIR),
    clip_threshold=0.22,
    story_backend='structured_template',
)
print(f'B complete: {B_OUTPUT_DIR}')


## 8. Run C

C는 독립 실행 셀입니다. C/D/E/F는 묶어서 실행하지 않습니다.


In [ ]:
clear_runtime_memory()
C_RESULT = run_selected_experiments(
    experiments=['c'],
    input_dir=str(STORY_INPUT_DIR),
    output_root=str(STORY_OUTPUT_ROOT),
)
print(f"C complete: {STORY_OUTPUT_ROOT / 'C'}")


## 9. Run D

D는 독립 실행 셀입니다.


In [ ]:
clear_runtime_memory()
D_RESULT = run_selected_experiments(
    experiments=['d'],
    input_dir=str(STORY_INPUT_DIR),
    output_root=str(STORY_OUTPUT_ROOT),
)
print(f"D complete: {STORY_OUTPUT_ROOT / 'D'}")


## 10. Run E

E는 독립 실행 셀입니다.


In [ ]:
clear_runtime_memory()
E_RESULT = run_selected_experiments(
    experiments=['e'],
    input_dir=str(STORY_INPUT_DIR),
    output_root=str(STORY_OUTPUT_ROOT),
)
print(f"E complete: {STORY_OUTPUT_ROOT / 'E'}")


## 11. Run F

F는 독립 실행 셀입니다. 현재 그림 기준 grounding 평가/기록이 포함된 실험입니다.


In [ ]:
clear_runtime_memory()
F_RESULT = run_selected_experiments(
    experiments=['f'],
    input_dir=str(STORY_INPUT_DIR),
    output_root=str(STORY_OUTPUT_ROOT),
)
print(f"F complete: {STORY_OUTPUT_ROOT / 'F'}")


## 12. 결과 요약

각 실험 결과 폴더에 생성된 JSON/TXT/HTML 수를 확인합니다.


In [ ]:
import pandas as pd

summary_rows = []
for exp in ['A', 'B', 'C', 'D', 'E', 'F']:
    folder = STORY_OUTPUT_ROOT / exp
    summary_rows.append({
        'experiment': exp,
        'folder': str(folder),
        'exists': folder.exists(),
        'json_count': len(list(folder.rglob('*.json'))) if folder.exists() else 0,
        'txt_count': len(list(folder.rglob('*.txt'))) if folder.exists() else 0,
        'html_count': len(list(folder.rglob('*.html'))) if folder.exists() else 0,
    })
summary_df = pd.DataFrame(summary_rows)
display(summary_df)


In [ ]:
def preview_text_files(experiment: str):
    folder = STORY_OUTPUT_ROOT / experiment.upper()
    files = sorted(list(folder.rglob('*.txt')))
    if not files:
        print(f'No txt files in {folder}')
        return
    for file_path in files[:3]:
        print('---', file_path.relative_to(PROJECT_DIR), '---')
        print(file_path.read_text(encoding='utf-8')[:1500])

preview_text_files('B')


## 13. 피드백 기록

피드백은 재생성에 사용하지 않고 평가/기록용으로만 저장합니다. 장면별 피드백과 전체 이야기 피드백을 모두 지원합니다.


In [ ]:
from datetime import datetime, timezone
from collections import defaultdict

FEEDBACK_DIR = STORY_OUTPUT_ROOT / 'feedback'
FEEDBACK_DIR.mkdir(parents=True, exist_ok=True)
FEEDBACK_RECORDS = FEEDBACK_DIR / 'feedback_records.jsonl'
FEEDBACK_SUMMARY = FEEDBACK_DIR / 'feedback_summary.json'

def _append_jsonl(path: Path, payload: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('a', encoding='utf-8') as f:
        f.write(json.dumps(payload, ensure_ascii=False) + '\n')

def _load_feedback_records():
    if not FEEDBACK_RECORDS.exists():
        return []
    return [json.loads(line) for line in FEEDBACK_RECORDS.read_text(encoding='utf-8').splitlines() if line.strip()]

def _write_feedback_summary():
    records = _load_feedback_records()
    grouped = defaultdict(lambda: {
        'scene_feedback_count': 0,
        'overall_feedback_count': 0,
        'scene_ratings': [],
        'overall_ratings': [],
    })
    for record in records:
        item = grouped[record.get('experiment', 'unknown')]
        if record.get('feedback_type') == 'scene':
            item['scene_feedback_count'] += 1
            if record.get('rating') is not None:
                item['scene_ratings'].append(record['rating'])
        if record.get('feedback_type') == 'overall':
            item['overall_feedback_count'] += 1
            if record.get('overall_rating') is not None:
                item['overall_ratings'].append(record['overall_rating'])
    summary = {'story_folder': SELECTED_STORY, 'total_records': len(records), 'by_experiment': {}}
    for experiment, item in grouped.items():
        summary['by_experiment'][experiment] = {
            'scene_feedback_count': item['scene_feedback_count'],
            'overall_feedback_count': item['overall_feedback_count'],
            'avg_scene_rating': sum(item['scene_ratings']) / len(item['scene_ratings']) if item['scene_ratings'] else None,
            'avg_overall_rating': sum(item['overall_ratings']) / len(item['overall_ratings']) if item['overall_ratings'] else None,
        }
    FEEDBACK_SUMMARY.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
    return summary

experiment_widget = widgets.Dropdown(options=['A', 'B', 'C', 'D', 'E', 'F'], description='실험:')
scene_widget = widgets.IntSlider(value=1, min=1, max=max(len(IMAGE_FILES), 1), step=1, description='장면:')
scene_rating_widget = widgets.IntSlider(value=3, min=1, max=5, step=1, description='장면 점수:')
scene_comment_widget = widgets.Textarea(description='장면 코멘트:', layout=widgets.Layout(width='720px', height='90px'))
overall_rating_widget = widgets.IntSlider(value=3, min=1, max=5, step=1, description='전체 점수:')
overall_comment_widget = widgets.Textarea(description='전체 코멘트:', layout=widgets.Layout(width='720px', height='110px'))
status_output = widgets.Output()

def save_scene_feedback(_button):
    scene_index = int(scene_widget.value)
    image_file = IMAGE_FILES[scene_index - 1].name if 1 <= scene_index <= len(IMAGE_FILES) else None
    payload = {
        'feedback_type': 'scene',
        'story_folder': SELECTED_STORY,
        'experiment': experiment_widget.value,
        'scene_index': scene_index,
        'image_file': image_file,
        'rating': int(scene_rating_widget.value),
        'comment': scene_comment_widget.value,
        'overall_rating': None,
        'overall_comment': None,
        'created_at': datetime.now(timezone.utc).isoformat(),
    }
    _append_jsonl(FEEDBACK_RECORDS, payload)
    summary = _write_feedback_summary()
    with status_output:
        status_output.clear_output()
        print('장면 피드백 저장 완료:', FEEDBACK_RECORDS)
        print(json.dumps(summary, ensure_ascii=False, indent=2))

def save_overall_feedback(_button):
    payload = {
        'feedback_type': 'overall',
        'story_folder': SELECTED_STORY,
        'experiment': experiment_widget.value,
        'scene_index': None,
        'image_file': None,
        'rating': None,
        'comment': None,
        'overall_rating': int(overall_rating_widget.value),
        'overall_comment': overall_comment_widget.value,
        'created_at': datetime.now(timezone.utc).isoformat(),
    }
    _append_jsonl(FEEDBACK_RECORDS, payload)
    summary = _write_feedback_summary()
    with status_output:
        status_output.clear_output()
        print('전체 이야기 피드백 저장 완료:', FEEDBACK_RECORDS)
        print(json.dumps(summary, ensure_ascii=False, indent=2))

save_scene_button = widgets.Button(description='장면 피드백 저장', button_style='info')
save_overall_button = widgets.Button(description='전체 피드백 저장', button_style='success')
save_scene_button.on_click(save_scene_feedback)
save_overall_button.on_click(save_overall_feedback)

display(experiment_widget)
display(scene_widget, scene_rating_widget, scene_comment_widget, save_scene_button)
display(overall_rating_widget, overall_comment_widget, save_overall_button)
display(status_output)


## 14. 피드백 요약과 다운로드


In [ ]:
summary = _write_feedback_summary()
print(json.dumps(summary, ensure_ascii=False, indent=2))
if FEEDBACK_RECORDS.exists():
    display(pd.DataFrame(_load_feedback_records()).tail(20))


In [ ]:
import shutil
from datetime import datetime

zip_base = PROJECT_DIR / f"{SELECTED_STORY}_outputs_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
zip_path = shutil.make_archive(str(zip_base), 'zip', root_dir=STORY_OUTPUT_ROOT)
print('Created:', zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    print('Not running in Colab. Download manually:', zip_path)
